<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit0/w08-docs-tests-readability/notebook.ipynb)


In [1]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

Colab detected — fetching the course (about 20 seconds)…
ready — the course is at /content/dev3pack


In [2]:
from bootcamp_agent.checks import check, review
from bootcamp_agent.hints import hint  # noqa: F401 - hint("w08-e1") when you want a nudge
import bootcamp_agent.week0_checks  # noqa: F401 — importing is what registers them

# Unit 8 — Documentation, tests and readability

**Week 0 · Course A, chapter 4 of 4 · about 60 minutes**

**Goal:** Write a docstring whose example `doctest` can run, take a `pytest` file from red to green by reading the assertion, and rename a function so the reader does not have to work out what it means.

**Why it matters:** `uv run pytest` is how this course decides whether your work is done, and a reviewer reading an assistant's diff is the last line of defence. Both are this unit.

Some cells below ship **broken on purpose**, marked `<------ EDIT THIS LINE`. Run them first and
read what happens. Debugging something wrong teaches more than filling in a blank.

## 1. A docstring doctest can run

**Context.** A docstring has a summary line, `:param` and `:return` lines, and an example after
`>>>`. The example is not decoration: `doctest` runs it and compares the output. The function below
is called `square`, its docstring says `square(3)` is `9`, and its body returns a cube.

**Instructions.**

1. Run the cell. `doctest` reports the failing example: expected 9, got 27.
2. Decide which is wrong, the example or the body. The function is called `square`.
3. Fix the body and run again. A passing doctest prints nothing.

**Expected output**

```
square(2) = 4
✅ w08-e1 passed
```

In [9]:
import doctest


def square(x):
    """Square the number x

    :param x: number to square
    :return: x squared

    >>> square(3)
    9
    """
    return x** 2   # <------ EDIT THIS LINE: the docstring says 9


# Run the docstring's own example as a test. It prints nothing when it passes.
doctest.run_docstring_examples(square, {"square": square}, name="square")
print(f"square(2) = {square(2)}")

square(2) = 4


In [10]:
check("w08-e1", square)

✅ w08-e1 passed


True

## 2. Red, then green

**Context.** `pytest` collects every `test_*.py` file and runs every `test_*` function in it. A test
is an `assert`. When one fails, pytest prints both sides of the comparison, and reading that diff
is the skill: the failure below is in the test, not in the `Document` class it tests.

**Instructions.**

1. Run the cell. One test fails; read the `AssertionError` and count the vowels on each side.
2. Fix the assertion in the test so it claims what the code does.
3. Run again until pytest reports `2 passed`.

**Expected output**

```
..                                                                       [100%]
2 passed in 0.01s
✅ w08-e2 passed
```

In [12]:
import subprocess
import sys
import tempfile
from pathlib import Path

work_dir = Path(tempfile.mkdtemp())
(work_dir / "tests").mkdir()
test_file = work_dir / "tests" / "test_document.py"

# work_dir/tests/test_document.py
test_file.write_text('''\
from collections import Counter
import re


class Document:
    def __init__(self, text):
        self.text = text
        self.tokens = self._tokenize()
        self.word_counts = Counter(self.tokens)

    def _tokenize(self):
        return re.findall(r"[a-zA-Z]+", self.text)


# Test tokens attribute on Document object
def test_document_tokens():
    doc = Document('a e i o u')
    assert doc.tokens == ['a', 'e', 'i', 'o','u']   # <------ EDIT THIS LINE: count the vowels


# Test edge case of empty document
def test_document_empty():
    doc = Document('')
    assert doc.tokens == []
    assert doc.word_counts == Counter()
''')

# working in the terminal: pytest tests/test_document.py
result = subprocess.run([sys.executable, "-m", "pytest", "tests/test_document.py", "-q",
                         "--color=no", "-p", "no:cacheprovider"],
                        cwd=work_dir, capture_output=True, text=True)
print(result.stdout.strip())

..                                                                       [100%]
2 passed in 0.01s


In [13]:
check("w08-e2", test_file)

✅ w08-e2 passed


True

## 3. Readability counts

**Context.** "Readability counts" is a line of the Zen of Python (`import this`). The lesson's bad
example is `check(x, y=100)`: it works, and nobody can say what it checks without reading the body.
In this notebook the name `check` belongs to the checker, so the starter is `chk`, which is no
better. The lesson's overdone example, `check_if_temperature_is_above_boiling_point(...)`, is the
other failure mode.

**Instructions.**

1. Run the cell. It works, and the printed names say nothing.
2. Rename the function and both parameters so they say what they are: a temperature, a boiling
   point, and a yes-or-no answer. Keep the default of 100 and the `>=` comparison.
3. Point `renamed` at the new function name so the check can find it.

**Expected output**

```
100 degrees -> True
99.9 degrees -> False
named 'is_boiling', parameters ['temp', 'boiling_point']
✅ w08-e3 passed
```

In [18]:
def is_boiling (temp, boiling_point=100):   # <------ EDIT THIS LINE: name the function and its parameters
    return temp >= boiling_point


renamed = is_boiling   # <------ EDIT THIS LINE: the new name


print(f"100 degrees -> {renamed(100)}")
print(f"99.9 degrees -> {renamed(99.9)}")
print(f"named {renamed.__name__!r}, parameters {list(renamed.__code__.co_varnames[:2])}")

100 degrees -> True
99.9 degrees -> False
named 'is_boiling', parameters ['temp', 'boiling_point']


In [19]:
check("w08-e3", renamed)

✅ w08-e3 passed


True

## Review

The scorecard for this unit. Every ❌ line names the exercise and the fix.

In [20]:
review("w08")

w08: 3/3 passed  ·  300/300 marks


True